# FRED-MD Group 6: Colab setup

Use the team's shared seed-1 package for the complete interest-and-exchange-rate group: 22 auxiliaries plus INDPRO.

1. In Colab, select a CPU runtime with no hardware accelerator.
2. Run the cells from top to bottom.
3. When prompted, upload fredmd-team-setup-seed1.zip, the original team ZIP.

This notebook creates an isolated Python 3.11 environment and installs the unchanged requirements.lock. It runs artifact verification, the seven existing tests, and three short group-6 smoke checks. It does not automatically run the 12 research experiments or evaluate test-set errors.

The Colab kernel is only a launcher. All study commands use the isolated environment, rather than Colab's preinstalled scientific packages. CPU, float32, and one-thread settings come from the shared study code.

Outputs are backed up to MyDrive/FREDMD_Group6_Seed1. The original ZIP is cached there for reconnects. Package files and frozen research inputs remain unchanged.

## 1. Mount Drive and choose working folders

Approve Colab's Drive access when prompted. The installed environment stays in session storage; results and logs go to Drive.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
import uuid
import zipfile

from google.colab import drive, files

drive.mount("/content/drive")

PROJECT = Path("/content/fredmd-group-curriculum-study")
DRIVE_FOLDER = Path("/content/drive/MyDrive/FREDMD_Group6_Seed1")
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
BACKUP_FOLDER = DRIVE_FOLDER / "sessions" / SESSION_ID
PYTHON = PROJECT / ".venv" / "bin" / "python"
SMOKES_READY = False

print("Study group: 6 (interest and exchange rates)")
print("Working folder:", PROJECT)
print("Session backup:", BACKUP_FOLDER)
print("Colab launcher:", sys.version.split()[0])


Mounted at /content/drive
Study group: 6 (interest and exchange rates)
Working folder: /content/fredmd-group-curriculum-study
Session backup: /content/drive/MyDrive/FREDMD_Group6_Seed1/sessions/20261031T203006Z_154216
Colab launcher: 3.13.18


## 2. Upload and extract the shared ZIP

On the first run, select the original team ZIP. On later sessions, the saved Drive copy is reused. Existing working files are checked rather than overwritten. Preparation is already complete; this notebook never calls study.py prepare.

In [2]:
EXPECTED_ZIP_SHA256 = "abed5b3cac9d7cc5f6a169b90c90e8afd3eb9e385f3867fe2c10f3a99b9dd83a"
ARCHIVE = DRIVE_FOLDER / "fredmd-team-setup-seed1.zip"

if not ARCHIVE.exists():
    print("Upload the original fredmd-team-setup-seed1.zip.")
    uploaded = files.upload()
    candidates = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one ZIP, then rerun this cell.")
    name, archive_bytes = candidates[0]
    if hashlib.sha256(archive_bytes).hexdigest() != EXPECTED_ZIP_SHA256:
        raise ValueError("This is not the reviewed team ZIP. Keep the original package and check its version.")
    ARCHIVE.write_bytes(archive_bytes)
    del uploaded, candidates, archive_bytes

archive_hash = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
if archive_hash != EXPECTED_ZIP_SHA256:
    raise ValueError("The Drive ZIP checksum differs from the reviewed package.")

if not PROJECT.exists():
    with zipfile.ZipFile(ARCHIVE) as archive:
        for entry in archive.infolist():
            candidate = (PROJECT.parent / entry.filename).resolve()
            if not candidate.is_relative_to(PROJECT.resolve()):
                raise ValueError("Unexpected ZIP path: " + entry.filename)
            if stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Unexpected symbolic link: " + entry.filename)
        archive.extractall(PROJECT.parent)

freeze = json.loads((PROJECT / "artifacts/freeze.json").read_text())
for relative, expected_hash in freeze["files"].items():
    actual_hash = hashlib.sha256((PROJECT / relative).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError("Frozen file differs: " + relative + ". Preserve outputs and check the working copy.")

print("ZIP checksum verified.")
print("Frozen scientific files verified:", len(freeze["files"]))
print("Existing prepared data will be reused.")


Upload the original fredmd-team-setup-seed1.zip.


Saving fredmd-team-setup-seed1.zip to fredmd-team-setup-seed1.zip
ZIP checksum verified.
Frozen scientific files verified: 24
Existing prepared data will be reused.


## 3. Install the isolated study environment

This may take several minutes. The Linux PyTorch wheel can download large dependencies even though the study runs on CPU. The dependency lock is kept unchanged. If installation fails, the log and error are saved; do not substitute package versions.

In [3]:
SETUP_LOGS = PROJECT / "results" / "setup" / SESSION_ID
SETUP_LOGS.mkdir(parents=True, exist_ok=True)

def backup_outputs():
    """Copy results and reports into this session's Drive backup."""
    BACKUP_FOLDER.mkdir(parents=True, exist_ok=True)
    for name in ("results", "reports"):
        source = PROJECT / name
        if source.exists():
            shutil.copytree(source, BACKUP_FOLDER / name, dirs_exist_ok=True)

def run_logged(command, label, extra_env=None):
    """Run a subprocess from the project, stream its log, and fail on errors."""
    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.update({
        "PYTHONNOUSERSITE": "1",
        "MPLBACKEND": "Agg",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
    })
    if extra_env:
        environment.update(extra_env)
    log_path = SETUP_LOGS / (label + ".log")
    print("\nRunning:", " ".join(map(str, command)), flush=True)
    with log_path.open("w") as handle:
        handle.write("COMMAND: " + json.dumps(list(map(str, command))) + "\n")
        with subprocess.Popen(
            list(map(str, command)),
            cwd=PROJECT,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                handle.write(line)
                handle.flush()
            return_code = process.wait()
    if return_code != 0:
        backup_outputs()
        raise RuntimeError(
            f"{label} failed with exit code {return_code}. "
            f"See {log_path.name} in the Drive session backup; keep the lock unchanged."
        )

bootstrap_environment = {
    "UV_PYTHON_INSTALL_DIR": str(PROJECT / ".colab_runtime" / "python"),
    "UV_CACHE_DIR": str(PROJECT / ".colab_runtime" / "uv_cache"),
    "UV_LINK_MODE": "copy",
}

try:
    uv = shutil.which("uv")
    if uv is None:
        run_logged([sys.executable, "-m", "pip", "install", "uv"], "install_uv")
        uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uv was installed but its executable was not found. Check the installation log.")

    if not PYTHON.exists():
        run_logged(
            [uv, "venv", "--python", "3.11", str(PROJECT / ".venv")],
            "create_python311",
            bootstrap_environment,
        )

    interpreter = json.loads(subprocess.check_output(
        [str(PYTHON), "-c", "import json,sys; print(json.dumps(list(sys.version_info[:2])))"],
        text=True,
    ))
    if interpreter != [3, 11]:
        raise RuntimeError("The study environment must use Python 3.11.")

    run_logged(
        [uv, "pip", "install", "--python", str(PYTHON), "-r", "requirements.lock"],
        "install_locked_dependencies",
        bootstrap_environment,
    )
    run_logged(
        [uv, "pip", "check", "--python", str(PYTHON)],
        "dependency_check",
        bootstrap_environment,
    )
finally:
    backup_outputs()

print("\nIsolated study environment:", PYTHON)



Running: /usr/local/bin/uv venv --python 3.11 /content/fredmd-group-curriculum-study/.venv
Using CPython 3.11.17
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate

Running: /usr/local/bin/uv pip install --python /content/fredmd-group-curriculum-study/.venv/bin/python -r requirements.lock
Resolved 93 packages in 4.01s
Prepared 92 packages in 2m 41s
Installed 93 packages in 48.84s
 + absl-py==2.5.1
 + annotated-types==0.8.0
 + anyio==4.15.1
 + basicts==1.1.0 (from file:///content/fredmd-group-curriculum-study/vendor/BasicTS)
 + blosc2==4.14.1
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + cloudpickle==3.1.2
 + contourpy==1.3.3
 + cycler==0.12.1
 + easy-torch==1.3.3
 + easydict==1.13
 + et-xmlfile==2.0.0
 + filelock==4.2.6
 + fonttools==4.68.0
 + fsspec==2026.9.0
 + grpcio==1.85.1
 + h11==0.16.0
 + h2==4.4.1
 + hf-xet==1.8.0
 + hpack==4.2.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==0.36.2
 + hyperframe==6.1.0
 + idna==3.20
 + iniconfig=

## 4. Verify the frozen setup and run the existing tests

Expected verification: 79 series; group 6 has 22 auxiliaries; INDPRO has 465 training, 118 validation, and 118 test windows. The package uses ddof=0. The seven tests check transformations, chronology, exposure, schedules, random streams, zero-valued targets, and model shapes.

In [4]:
try:
    run_logged([PYTHON, "study.py", "verify"], "verify_frozen_setup")
    run_logged([PYTHON, "-m", "pytest", "tests", "-q"], "existing_tests")
finally:
    backup_outputs()

prepared_metadata = json.loads((PROJECT / "data/prepared/metadata.json").read_text())
if len(prepared_metadata["series"]) != 79 or len(prepared_metadata["groups"]["6"]) != 22:
    raise RuntimeError("Unexpected prepared group membership.")

print("\nFrozen inputs and tests passed.")
print("INDPRO windows:", prepared_metadata["series"]["INDPRO"]["window_counts"])
print("Auxiliary series:", prepared_metadata["groups"]["6"])



Running: /content/fredmd-group-curriculum-study/.venv/bin/python study.py verify
{'verified': True, 'series': 79, 'groups': {'1': 15, '2': 31, '3': 10, '6': 22}, 'indpro_windows': {'train': 465, 'val': 118, 'test': 118}, 'snapshot_sha256': 'd6e239038afbdbf4667c2a63c13c942817bc196fedab4355cd7e0d353ecdd65d'}

Running: /content/fredmd-group-curriculum-study/.venv/bin/python -m pytest tests -q
.......                                                                  [100%]
7 passed in 9.46s

Frozen inputs and tests passed.
INDPRO windows: {'train': 465, 'val': 118, 'test': 118}
Auxiliary series: ['AAA', 'AAAFFM', 'BAA', 'BAAFFM', 'COMPAPFFx', 'CP3Mx', 'EXCAUSx', 'EXJPUSx', 'EXSZUSx', 'EXUSUKx', 'FEDFUNDS', 'GS1', 'GS10', 'GS5', 'T10YFFM', 'T1YFFM', 'T5YFFM', 'TB3MS', 'TB3SMFFM', 'TB6MS', 'TB6SMFFM', 'TWEXAFEGSMTHx']


## 5. Run the three group-6 smoke checks

Each check trains for exactly three updates and validates, without evaluating test-set errors. These outputs are technical checks, not research results. Unique session folders make this cell safe to rerun without overwriting earlier outputs.

In [5]:
smoke_results = []
smoke_attempt = uuid.uuid4().hex[:6]

for model in ("linear", "mlp", "patchtst"):
    output = PROJECT / "results" / "smoke" / SESSION_ID / smoke_attempt / "group6" / model / "seed1"
    try:
        run_logged(
            [PYTHON, "study.py", "smoke", "--group", "6",
             "--model", model, "--output", output],
            "smoke_" + smoke_attempt + "_" + model,
        )
        summary = json.loads((output / "summary.json").read_text())
        if summary["updates"] != 3 or summary["smoke_only"] is not True:
            raise RuntimeError("Smoke run did not report exactly three updates.")
        if "test_mse" in summary or "test_mae" in summary:
            raise RuntimeError("A smoke check unexpectedly reported test metrics.")
        smoke_results.append(summary)
    finally:
        backup_outputs()

SMOKES_READY = True
print("\nAll three group-6 smoke checks passed.")
print("No full research run or test-set evaluation has been requested by this notebook.")



Running: /content/fredmd-group-curriculum-study/.venv/bin/python study.py smoke --group 6 --model linear --output /content/fredmd-group-curriculum-study/results/smoke/20261031T203006Z_154216/33f74a/group6/linear/seed1
2026-10-31 20:49:09,180 - easytorch-env - INFO - Use deterministic algorithms.
2026-10-31 20:49:09,180 - BasicTS - INFO - Building model.
2026-10-31 20:49:09,181 - BasicTS - INFO - Set ckpt save dir: "/content/fredmd-group-curriculum-study/results/smoke/20261031T203006Z_154216/33f74a/group6/linear/seed1/basicts/d1aca5013797f72ed111b828de129180"
2026-10-31 20:49:09,186 - BasicTS-study - INFO - Initializing training.
2026-10-31 20:49:09,186 - BasicTS-study - INFO - Building train data loader.
2026-10-31 20:49:15,036 - BasicTS-study - INFO - Set optim: Adam
2026-10-31 20:49:15,036 - BasicTS-study - INFO - Building val data loader.
2026-10-31 20:49:15,093 - BasicTS-study - INFO - Building test data loader.
2026-10-31 20:49:15,142 - BasicTS-study - INFO - Total parameters: 75

## 6. Save setup status and prepare the next step

The helper below will run one research condition only when you explicitly call it later. It preserves the package's training implementation and backs up outputs after each call.

In [6]:
runtime_probe = (
    "import importlib.metadata as m,json,platform,sys;"
    "print(json.dumps({'python':sys.version,'platform':platform.platform(),"
    "'machine':platform.machine(),'processor':platform.processor(),"
    "'versions':{n:m.version(n) for n in "
    "['BasicTS','torch','numpy','pandas','easy-torch']}}))"
)
scientific_runtime = json.loads(subprocess.check_output([str(PYTHON), "-c", runtime_probe], text=True))

status = {
    "session_id": SESSION_ID,
    "group": 6,
    "auxiliary_series": prepared_metadata["groups"]["6"],
    "archive_sha256": archive_hash,
    "freeze_sha256": hashlib.sha256((PROJECT / "artifacts/freeze.json").read_bytes()).hexdigest(),
    "runtime": scientific_runtime,
    "smoke_checks": [
        {"model": row["model"], "updates": row["updates"], "smoke_only": row["smoke_only"]}
        for row in smoke_results
    ],
    "research_runs_started_by_setup": 0,
}
status_path = PROJECT / "reports" / ("colab_setup_" + SESSION_ID + ".json")
status_path.write_text(json.dumps(status, indent=2) + "\n")
backup_outputs()

def run_group6(model, order):
    """Explicitly start one 800-update research run and preserve its outputs."""
    if not SMOKES_READY:
        raise RuntimeError("Complete all three smoke checks before research runs.")
    if model not in ("linear", "mlp", "patchtst"):
        raise ValueError("Choose linear, mlp, or patchtst.")
    if order not in ("random", "auxiliary_first", "target_first", "interleaved"):
        raise ValueError("Choose one of the four agreed schedules.")
    output = PROJECT / "results" / "study" / "group6" / order / model / "seed1"
    if output.exists():
        raise FileExistsError("That run directory already exists. Preserve it before planning a rerun.")
    try:
        run_logged(
            [PYTHON, "study.py", "run", "--group", "6", "--model", model,
             "--order", order, "--output", output],
            "research_" + model + "_" + order,
        )
        return json.loads((output / "summary.json").read_text())
    finally:
        backup_outputs()

print("SETUP COMPLETE")
print("Study Python:", scientific_runtime["python"].split()[0])
print("Pinned scientific versions:", scientific_runtime["versions"])
print("Drive backup:", BACKUP_FOLDER)
print("Your next research condition is Linear / random. It has NOT been started.")


SETUP COMPLETE
Study Python: 3.11.17
Pinned scientific versions: {'BasicTS': '1.1.0', 'torch': '2.5.1', 'numpy': '1.26.4', 'pandas': '2.2.3', 'easy-torch': '1.3.3'}
Drive backup: /content/drive/MyDrive/FREDMD_Group6_Seed1/sessions/20261031T203006Z_154216
Your next research condition is Linear / random. It has NOT been started.


## After setup

Stop when the final cell prints SETUP COMPLETE. The setup has not run your 12 research experiments.

When you are ready to start the first full condition, add a new code cell containing:

```python
first_result = run_group6("linear", "random")
first_result
```

That command performs the full 800-update condition and evaluates the final model on INDPRO. Subsequent research commands should follow the agreed plan.

On a fresh Colab runtime, rerun this notebook from the top. The original ZIP is reused from Drive, while Python and dependencies are recreated in session storage. Previous session outputs remain in the Drive sessions folder. Your scientific files and shared package are not regenerated.

## References

- [Google Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)
- [uv Python environments](https://docs.astral.sh/uv/pip/environments/)
- [uv managed Python installation](https://docs.astral.sh/uv/guides/install-python/)

In [17]:
first_result = run_group6("linear", "random")
first_result

FileExistsError: That run directory already exists. Preserve it before planning a rerun.

In [ ]:
mlp_random = run_group6("mlp", "random")
mlp_random

In [ ]:
patchtst_random = run_group6("patchtst", "random")
patchtst_random

In [18]:
linear_aux_first = run_group6("linear", "auxiliary_first")
linear_aux_first

FileExistsError: That run directory already exists. Preserve it before planning a rerun.

In [19]:
mlp_aux_first = run_group6("mlp", "auxiliary_first")
mlp_aux_first

FileExistsError: That run directory already exists. Preserve it before planning a rerun.

In [ ]:
patchtst_aux_first = run_group6("patchtst", "auxiliary_first")
patchtst_aux_first

In [ ]:
linear_target_first = run_group6("linear", "target_first")
linear_target_first

In [20]:
mlp_target_first = run_group6("mlp", "target_first")
mlp_target_first

FileExistsError: That run directory already exists. Preserve it before planning a rerun.

In [ ]:
patchtst_target_first = run_group6("patchtst", "target_first")
patchtst_target_first

In [21]:
linear_interleaved = run_group6("linear", "interleaved")
mlp_interleaved = run_group6("mlp", "interleaved")
patchtst_interleaved = run_group6("patchtst", "interleaved")
linear_interleaved
mlp_interleaved
patchtst_interleaved

FileExistsError: That run directory already exists. Preserve it before planning a rerun.

In [23]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    str(DRIVE_FOLDER / f"group6_results_seed1_{SESSION_ID}"),
    "zip",
    root_dir=str(PROJECT / "results" / "study"),
    base_dir="group6",
)

files.download(archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>